In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Fine-Tuning Qwen2.5-1.5B untuk Bahasa Indonesia

# Basic

## Persiapan Dependencies dan GPU

In [2]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install -q \
    "unsloth==2026.9.4" \
    "transformers==4.56.2" \
    "trl==0.22.2" \
    "datasets==4.3.0" \
    "bitsandbytes==0.50.2" \
    "xformers==0.0.34" \
    "peft==0.20.0" \
    "accelerate" \
    "sentencepiece" \
    "protobuf" \
    "hf_transfer"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.0
    Uninstalling datasets-5.0.0:
      Successfully uninstalled datasets-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 17.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from transformers import TextStreamer
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import wandb


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1554: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU Detected: {gpu_stats.name}. Max Memory = {max_memory} GB.")
else:
    print("WARNING: GPU Not Detected.")

GPU Detected: Tesla T4. Max Memory = 14.562 GB.


## Panggil Model dan Tokenizer

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B",
    max_seq_length=1024,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    dtype=None,
)

==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 1.400 GiB
no_split classes   : ['Qwen2DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 12.108 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.98 GiB  weights  0.856 GiB  free 12.122 GiB  reserve 12.108 GiB
  cuda:1  budget  13.00 GiB  weights  0.543 GiB  free 12.453 GiB  reserve 11.938 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.421 -> 12.979 GiB, cuda:1 14.440 -> 12.996 GiB (memory th

In [21]:
print(model.config.quantization_config)

{'bnb_4bit_compute_dtype': torch.float16, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.0.self_attn', 'model.layers.0.mlp', 'model.layers.1.mlp', 'model.layers.2.mlp', 'model.layers.26.mlp', 'model.layers.15.self_attn'], 'llm_int8_threshold': 6.0, 'load_in_4bit': True, 'load_in_8bit': False, 'quant_method': 'bitsandbytes'}


In [7]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5"
)

## Persiapan Dataset 

Menggunakan fungsi mapping dari Hugging Face datasets untuk mengubah dataset mentah mereka ke dalam standar Chat Template yang didukung Unsloth.

In [9]:
dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split = "train")
print(f"Jumlah data training: {len(dataset)}")

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Jumlah data training: 49969


In [10]:
system_prompt = "Kamu adalah asisten AI yang membantu menjawab pertanyaan pengguna berdasarkan konteks yang diberikan."

def formatting_prompts_func(examples):
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []


    for input, output in zip(inputs, outputs):
        user_content = f"\n\nContext:\n{input}"
        conversation = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output}
        ]
        text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }


dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/49969 [00:00<?, ? examples/s]

In [11]:
dataset[50]["text"]

'<|im_start|>system\nKamu adalah asisten AI yang membantu menjawab pertanyaan pengguna berdasarkan konteks yang diberikan.<|im_end|>\n<|im_start|>user\n\n\nContext:\nTuliskan pernyataan satu kalimat yang akurat mencerminkan ide utama dari bagian berikut.\nDalam studi terbaru, disarankan bahwa pemanasan global memiliki dampak yang lebih signifikan pada lingkungan daripada yang telah diantisipasi sebelumnya.<|im_end|>\n<|im_start|>assistant\nStudi terbaru menyarankan bahwa dampak pemanasan global pada lingkungan lebih signifikan dibandingkan dengan yang sebelumnya diantisipasi.<|im_end|>\n'

In [20]:
print(dataset[50]["text"])

<|im_start|>system
Kamu adalah asisten AI yang membantu menjawab pertanyaan pengguna berdasarkan konteks yang diberikan.<|im_end|>
<|im_start|>user


Context:
Tuliskan pernyataan satu kalimat yang akurat mencerminkan ide utama dari bagian berikut.
Dalam studi terbaru, disarankan bahwa pemanasan global memiliki dampak yang lebih signifikan pada lingkungan daripada yang telah diantisipasi sebelumnya.<|im_end|>
<|im_start|>assistant
Studi terbaru menyarankan bahwa dampak pemanasan global pada lingkungan lebih signifikan dibandingkan dengan yang sebelumnya diantisipasi.<|im_end|>



## Konfigurasi LoRA

Memanfaatkan teknik QLoRA saat memuat model dengan mengaktifkan double quantization dalam format 4-bit. Siswa wajib mendefinisikan Adapter LoRA pada setidaknya salah satu komponen komputasi utama model. Multi-Head Attention Feed Forward Network

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

Unsloth 2026.9.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Menjalankan SFTTrainer pada model text generation, minimal selama 800 steps hingga selesai tanpa terjadi error VRAM (OOM).

In [13]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="./outputs",

    # Training
    num_train_epochs=1,
    max_steps=800,

    # Batch size
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # Memory optimization
    gradient_checkpointing=True,

    # Learning rate
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,

    # Precision
    fp16=True,
    bf16=False,

    # Dataset
    dataset_text_field="text",
    max_length=1024,
    packing=True,

    # Optimizer
    optim="paged_adamw_8bit",
    weight_decay=0.01,

    # Logging
    logging_steps=10,
    report_to="wandb",

    # Save
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,

    # Reproducibility
    seed=3407,
)

In [15]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=sft_config
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/49969 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/49969 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [16]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 13,740 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.155400
20,1.872300
30,1.633600
40,1.487200
50,1.461900
60,1.396600
70,1.419800
80,1.423400
90,1.362600
100,1.413300


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run


TrainOutput(global_step=800, training_loss=1.3626443827152253, metrics={'train_runtime': 2929.7866, 'train_samples_per_second': 1.092, 'train_steps_per_second': 0.273, 'total_flos': 2.5965888200011776e+16, 'train_loss': 1.3626443827152253, 'epoch': 0.2328966521106259})

In [17]:
gpu_stats = torch.cuda.get_device_properties(0)

print(f"GPU: {gpu_stats.name}")
print(f"Max Memory: {gpu_stats.total_memory / 1024**3:.2f} GB")
print(
    f"Peak reserved memory: "
    f"{torch.cuda.max_memory_reserved() / 1024**3:.2f} GB"
)

GPU: Tesla T4
Max Memory: 14.56 GB
Peak reserved memory: 1.97 GB


Mengunggah model hasil Fine-tuning Anda ke repositori Hugging Face agar dapat dipanggil kembali pada tahap inferensi RAG. Gunakan perintah model.push_to_hub dengan metode merged_16bit.

In [19]:
model.push_to_hub_merged(
    "alaqsaka/fine-tuning-slm-legal-chatbot-using-qwen",
    tokenizer,
    save_method="merged_16bit",
    token=os.environ["HF_TOKEN"]
)

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

... (3 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:51<00:00, 51.22s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/alaqsaka/fine-tuning-slm-legal-chatbot-using-qwen`
